In [7]:
import os
import datetime
import folium
import xarray as xr
import streamlit as st
from streamlit_folium import st_folium
from multiprocessing import Pool

from plotting.functions.plot_EFD import plot_EFD, aggregate_EFD_angles, aggregate_EFD_waveform
from plotting.functions.plot_LAP import lap_plot, aggregated_LAP_electron
from plotting.functions.plot_HEPPX import heppx_plot, aggregated_HEPPX_xray
from plotting.functions.plot_SCM import scmplot, aggregated_SCM_angles, aggregated_SCM_waveform
from plotting.functions.plot_sequential_SCM import plot_sequential_SCM
from plotting.functions.plot_sequential_EFD import plot_sequential_EFD
from plotting.functions.plot_sequential_HEPPL import plot_sequential_HEPPL
from plotting.functions.plot_sequential_HEPPX import plot_sequential_HEPPX
from plotting.functions.HEPPH_MUL_plot import plot_sequential_HEPPH
from plotting.functions.HEPD_V2_fixed import plot_HEPPD
from plotting.functions.HEPPD_Mul_plot import plot_HEPD_multiple_files
from plotting.functions.LAP_Mul_plot_Final import plot_sequential_LAP
from plotting.functions.plot_HEPPL import aggregated_HEPPL_electron_proton, plot_hepl
from plotting.functions.plot_HEPPH import plotheph, aggregated_HEPPH_electron_proton
from plotting.functions.plot_HEPPD import aggregate_HEPPD_electron_proton

# Replace with your folder path
folder_path = '/home/grp2/dhruva-sharma/scientific-dashboard/webappfiles/data/concaat/'

def dataset(path):
    try:
        ds = xr.open_zarr(path)
        return ds
    except Exception as e:
        ds = xr.open_dataset(path, engine='h5netcdf', phony_dims='sort')
        return ds

def extract_dates(file_name):
    from datetime import datetime as dt

    try:
        base_name = os.path.basename(file_name)
        parts = base_name.split('_')
        
        start_index = None
        for i in range(len(parts)):
            if parts[i].isdigit() and len(parts[i]) == 8:
                start_index = i
                break
        
        if start_index is None:
            raise ValueError(f"Formato data non trovato nel nome del file: {file_name}")
        
        start_date_str = '_'.join(parts[start_index:start_index + 2])
        end_date_str = '_'.join(parts[start_index + 2:start_index + 4])  
        
        start_date = dt.strptime(start_date_str, '%Y%m%d_%H%M%S').date()
        end_date = dt.strptime(end_date_str, '%Y%m%d_%H%M%S').date()
        
        return start_date, end_date
    except ValueError as e:
        print(f"Errore nel parsing delle date per il file {file_name}: {e}")
        return None, None

def draw_map():
    m = folium.Map(location=[45.5236, -122.6750], zoom_start=1.15, continuousWorld=False, noWrap=True)
    draw = folium.plugins.Draw(
        draw_options={
            'polyline': False,
            'polygon': True,
            'circle': False,
            'marker': False,
            'circlemarker': False,
            'rectangle': True,
        }
    )
    draw.add_to(m)
    st_map = st_folium(m, width=1000, height=400)
    col1, col2 = st.columns(2)

    start_date = col1.date_input("Start date", datetime.date(2018, 2, 15))
    end_date = col2.date_input("End date", value=datetime.date.today())
    
    last_active_drawing = st_map.get('last_active_drawing')
    
    if last_active_drawing:
        geometry = last_active_drawing.get('geometry')
        if geometry and 'coordinates' in geometry:
            coordinates = geometry['coordinates'][0][:-1]

            pathskidibidi = polygon(coordinates, file_selector())
            pathskidibidi = check_Date_interval(pathskidibidi, start_date, end_date)
            dataset_type = extract_dataset_type(pathskidibidi)

            option = st.multiselect(
                "Instrument Type",
                (" ", "EFD", "SCM", "LAP", "HEP_1", "HEP_4", "HEP_2", "HEP_DDD"))

            if st.button("plot"):
                if option:
                    tasks = []
                    for dataset in dataset_type:
                        file_path = dataset[0]
                        ds_type = dataset[1]
                        
                        for sensor in option:
                            if ds_type == 'EFD' and sensor == 'EFD':
                                tasks.append((file_path, 'plot_EFD'))
                            elif ds_type == 'LAP' and sensor == 'LAP':
                                tasks.append((file_path, 'lap_plot'))
                            elif ds_type == 'HEP_4' and sensor == 'HEP_4':
                                tasks.append((file_path, 'heppx_plot'))
                            elif ds_type == 'HEP_1' and sensor == 'HEP_1':
                                tasks.append((file_path, 'plot_hepl'))
                            elif ds_type == 'SCM' and sensor == 'SCM':
                                tasks.append((file_path, 'scmplot'))
                            elif ds_type == 'HEP_2' and sensor == 'HEP_2':
                                tasks.append((file_path, 'plotheph'))
                            elif ds_type == 'HEP_DDD' and sensor == 'HEP_DDD':
                                tasks.append((file_path, 'plot_HEPPD'))

                    with Pool() as pool:
                        pool.map(paralleltest, tasks)
                        
                min_lat, max_lat = calculate_extremes(coordinates)
                st.write(f"Leftmost Latitude: {min_lat:.6f}")
                st.write(f"Rightmost Latitude: {max_lat:.6f}")
        else:
            st.write("No 'coordinates' found in 'geometry'.")
    else:
        st.write("No 'last_active_drawing' found in st_map.")

def paralleltest(args):
    file_path, plot_type = args
    if plot_type == 'plot_EFD':
        plot_EFD(file_path)
    elif plot_type == 'lap_plot':
        lap_plot(file_path)
    elif plot_type == 'heppx_plot':
        heppx_plot(file_path)
    elif plot_type == 'plot_hepl':
        plot_hepl(file_path)
    elif plot_type == 'scmplot':
        scmplot(file_path)
    elif plot_type == 'plotheph':
        plotheph(file_path)
    elif plot_type == 'plot_HEPPD':
        plot_HEPPD(file_path)

def polygon(points, files):    
    intersectionFile = []
    latitudes = [point[1] for point in points]
    longitudes = [point[0] for point in points]
 
    lat_min = min(latitudes)
    lat_max = max(latitudes)
    lon_min = min(longitudes)
    lon_max = max(longitudes)
 
    for file in files:
        ds = dataset(file)
        try:
            geo_lat = ds.GEO_LAT
            geo_lon = ds.GEO_LON
        except:
            LonLat = ds.LonLat
            if LonLat.ndim == 3:
                geo_lon = LonLat[0, 0, :]
                geo_lat = LonLat[0, 0, :]
            elif LonLat.ndim == 2:
                geo_lon = LonLat[:, 0]
                geo_lat = LonLat[:, 1]
        lat_mask = (geo_lat >= lat_min) & (geo_lat <= lat_max)
        lon_mask = (geo_lon >= lon_min) & (geo_lon <= lon_max)
 
        final_mask = lat_mask & lon_mask
 
        if final_mask.any():
            intersectionFile.append(file)
    return intersectionFile

def check_Date_interval(files_path, start_date_selector, end_date_selector):
    files_list = []
    for file in files_path:
        start_date, end_date = extract_dates(file)
        if start_date_selector <= end_date and end_date_selector >= start_date:
            files_list.append(file)
    return files_list

def extract_dataset_type(file_paths):    
    dataset_types = []
    
    for file_path in file_paths:
        file_name = os.path.basename(file_path)
        
        if 'EFD' in file_name:
            dataset_types.append([file_path, 'EFD'])
        elif 'LAP' in file_name:
            dataset_types.append([file_path, 'LAP'])
        elif 'SCM' in file_name:
            dataset_types.append([file_path, 'SCM'])
        elif 'HEP_4' in file_name:
            dataset_types.append([file_path, 'HEP_4'])
        elif 'HEP_1' in file_name:
            dataset_types.append([file_path, 'HEP_1'])
        elif 'HEP_2' in file_name:
            dataset_types.append([file_path, 'HEP_2'])
        elif 'HEP_DDD' in file_name:
            dataset_types.append([file_path, 'HEP_DDD'])
        else:
            dataset_types.append([file_path, 'UNKNOWN'])
    
    return dataset_types

def calculate_extremes(points):
    min_lat = min(point[1] for point in points)
    max_lat = max(point[1] for point in points)
    return min_lat, max_lat

def file_selector(folder_path=folder_path):
    file_paths = []
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".zarr") or file.endswith(".nc"):
                file_paths.append(os.path.join(root, file))
    return file_paths

if __name__ == "__main__":
    st.title("Scientific Dashboard")
    draw_map()


ModuleNotFoundError: No module named 'plotting.functions'